<a href="https://colab.research.google.com/github/Khan-Fazal-sys/Deep-Residual-Learning-for-Image-Recognition-/blob/main/Deep_Residual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import argparse

# -------------------------------
# 1. Residual Block (Basic)
# -------------------------------
class BasicBlock(nn.Module):
    """Basic residual block for CIFAR-10 (two 3x3 convs + shortcut)"""
    expansion = 1   # output channels = planes * expansion (1 for basic)

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        # First conv
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # Second conv
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        # Shortcut (identity or projection)
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

# -------------------------------
# 2. ResNet for CIFAR-10 (paper architecture)
# -------------------------------
class ResNet_CIFAR10(nn.Module):
    """
    ResNet for CIFAR-10 as described in Sec. 4.2 of the paper.
    Depth = 6n + 2, where n = number of blocks per stage.
    Default n=5 -> 32 layers. For 110 layers, n=18.
    """
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 16

        # Initial conv: 3x3, stride 1 (CIFAR-10 images are 32x32)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Three stages: feature map sizes 32 -> 16 -> 8
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)

        # Global average pooling + FC
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64 * block.expansion, num_classes)

        # Initialize weights (He init)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )
        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)

        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

# Helper function to create ResNet of given depth
def resnet_for_cifar10(depth: int, num_classes=10):
    """depth = 6n + 2, e.g., 20, 32, 44, 56, 110, 1202"""
    assert (depth - 2) % 6 == 0, "Depth must be 6n+2"
    n = (depth - 2) // 6
    num_blocks = [n, n, n]
    return ResNet_CIFAR10(BasicBlock, num_blocks, num_classes)

# -------------------------------
# 3. Training on CIFAR-10
# -------------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--depth', type=int, default=56, help='ResNet depth (6n+2)')
    parser.add_argument('--epochs', type=int, default=64, help='Number of epochs')
    parser.add_argument('--batch_size', type=int, default=128)
    parser.add_argument('--lr', type=float, default=0.1)
    parser.add_argument('--weight_decay', type=float, default=1e-4)
    parser.add_argument('--momentum', type=float, default=0.9)
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
    args = parser.parse_args([]) # Modified this line

    device = torch.device(args.device)
    print(f"Using device: {device}")

    # Data augmentation & normalization (same as paper)
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

    trainloader = DataLoader(trainset, batch_size=args.batch_size, shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=args.batch_size, shuffle=False, num_workers=2)

    # Build ResNet
    model = resnet_for_cifar10(args.depth, num_classes=10).to(device)
    print(f"ResNet-{args.depth} created. Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[32, 48], gamma=0.1)  # divide LR at 32k and 48k iterations

    best_acc = 0.0
    for epoch in range(1, args.epochs + 1):
        train_loss, train_acc = train_one_epoch(model, trainloader, criterion, optimizer, device)
        test_loss, test_acc = validate(model, testloader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), f'resnet_{args.depth}_cifar10_best.pth')

    print(f"Best test accuracy: {best_acc:.2f}%")

if __name__ == '__main__':
    main()

Using device: cuda
ResNet-56 created. Number of parameters: 855,770
Epoch  1 | Train Loss: 1.9532 | Train Acc: 27.40% | Test Loss: 1.6363 | Test Acc: 38.94%
Epoch  2 | Train Loss: 1.4876 | Train Acc: 45.15% | Test Loss: 1.2781 | Test Acc: 54.12%
Epoch  3 | Train Loss: 1.1511 | Train Acc: 58.75% | Test Loss: 1.0867 | Test Acc: 62.41%
Epoch  4 | Train Loss: 0.9207 | Train Acc: 67.50% | Test Loss: 0.9254 | Test Acc: 67.95%
Epoch  5 | Train Loss: 0.7589 | Train Acc: 73.39% | Test Loss: 0.8236 | Test Acc: 73.22%
Epoch  6 | Train Loss: 0.6686 | Train Acc: 76.84% | Test Loss: 0.7752 | Test Acc: 75.52%
Epoch  7 | Train Loss: 0.5959 | Train Acc: 79.61% | Test Loss: 0.6751 | Test Acc: 77.94%
Epoch  8 | Train Loss: 0.5444 | Train Acc: 81.13% | Test Loss: 0.6657 | Test Acc: 77.84%
Epoch  9 | Train Loss: 0.5133 | Train Acc: 82.28% | Test Loss: 0.5338 | Test Acc: 81.62%
Epoch 10 | Train Loss: 0.4806 | Train Acc: 83.26% | Test Loss: 0.6596 | Test Acc: 77.92%
Epoch 11 | Train Loss: 0.4539 | Train Acc: